# 全参考评估脚本：PSNR / SSIM / LPIPS / 线条 F1

本 notebook 用于在 **同一测试集** 上，对不同模型的 **配对 SR/HR 图像** 做全参考评估：

- 失真类指标：**PSNR / SSIM**（基于像素与结构的还原度）
- 感知类指标：**LPIPS** （越低越好，1-LPIPS 越高越好）
- 动漫特化指标：**线条保真度 F1**（Canny 边缘检测 + F1-score）

已适配以下架构：
- 我们的 RRDB 23-block / 6-block（2× / 4×）
- APISR 官方 RRDB 2x / 4x 预训练模型

本 notebook 只做 **评估**，不进行训练，默认在本地/Colab 已经生成好 SR 图像或已准备好 LR/HR 及模型权重。

In [1]:
# ============================================================
# 0. 基本配置（本地多模型评估）
# ============================================================

import os
import glob
import json
from math import log10
import time

import cv2
import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F

from skimage.metrics import structural_similarity as ssim
import lpips
import sys

# RRDB 从 APISR_tools/architecture/rrdb.py 引入
apisr_tools_path = os.path.abspath('APISR_tools')
if apisr_tools_path not in sys.path:
    sys.path.insert(0, apisr_tools_path)
from architecture.rrdb import RRDBNet

def build_rrdb(scale, num_block):
    return RRDBNet(3, 3, scale=scale, num_block=num_block)

# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# 项目根目录（默认就是当前工作目录）
PROJECT_DIR = os.getcwd()

# HR / 结果目录（使用绝对路径，避免 Jupyter 工作目录变化带来的问题）
HR_DIR = os.path.join(PROJECT_DIR, 'dataset/highres/original')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# 试运行：只评估前 N 张图；设为 None 则评估全部
MAX_IMAGES = None

# 从第几个模型开始评估（0-based）。4 = 从第 5 个模型开始
START_MODEL_INDEX = 4

# 测试集名单：只评估该文件中的图像（每行一个文件名，共 434 张）
TEST_LIST_PATH = os.path.join(PROJECT_DIR, 'dataset/test_list.txt')

# APISR 官方 RRDB 预训练权重（相对于 PROJECT_DIR）
CKPT_APISR_2X = 'pretrained_models/2x_APISR_RRDB_GAN_generator.pth'
CKPT_APISR_4X = 'pretrained_models/4x_APISR_RRDB_GAN_generator.pth'

# 多个待评估模型配置（按顺序评估；不需要的可以注释或删掉）
MODEL_CONFIGS = [
    {
        'model_name': 'ESRGAN_4x_finetune',
        'scale': 4,
        'num_block': 23,
        'is_apisr': False,
        'ckpt_ours': 'saved_models/ESRGAN_4x_finetune_best.pth',
        'lr_dir': 'dataset/lowres_4x/original',
    },
    {
        'model_name': 'ESRGAN_2x_finetune',
        'scale': 2,
        'num_block': 6,
        'is_apisr': False,
        'ckpt_ours': 'saved_models/ESRGAN_2x_finetune_best.pth',
        'lr_dir': 'dataset/lowres_2x/original',
    },
    {
        'model_name': 'APISR_RRDB_4x',
        'scale': 4,
        'num_block': 6,
        'is_apisr': True,
        'ckpt_ours': None,
        'lr_dir': 'dataset/lowres_4x_simple/original',
    },
    {
        'model_name': 'APISR_RRDB_2x',
        'scale': 2,
        'num_block': 6,
        'is_apisr': True,
        'ckpt_ours': None,
        'lr_dir': 'dataset/lowres_2x_simple/original',
    },
    {
        'model_name': 'GRL_2x_finetune',
        'scale': 2,
        'model_type': 'grl',
        'ckpt_ours': 'saved_models/GRL_2x_finetune_best.pth',
        'lr_dir': 'dataset/lowres_2x/original',
    },
]

configs_to_run = MODEL_CONFIGS[START_MODEL_INDEX:]
print(f'共 {len(MODEL_CONFIGS)} 个模型；从第 {START_MODEL_INDEX + 1} 个开始，本轮评估 {len(configs_to_run)} 个（START_MODEL_INDEX={START_MODEL_INDEX}）。')
if MAX_IMAGES is not None:
    print(f'[试运行] 每个模型仅评估前 {MAX_IMAGES} 张图（改 MAX_IMAGES = None 可跑全量）。')


Device: cuda
本轮将依次评估 4 个模型。


In [2]:
# ============================================================
# 1. 构建并加载模型（按需）
# ============================================================

def load_rrdb_model(scale: int, num_block: int, is_apisr: bool, ckpt_ours: str | None = None):
    """根据配置加载 RRDB 模型（支持 APISR 官方权重和自训练权重）。"""
    model = build_rrdb(scale=scale, num_block=num_block).to(device)
    model.eval()

    # 选择权重路径
    if is_apisr:
        ckpt_path = CKPT_APISR_2X if scale == 2 else CKPT_APISR_4X
    else:
        ckpt_path = ckpt_ours

    if not ckpt_path:
        raise FileNotFoundError('未提供权重路径，请检查配置中的 ckpt_ours 或 CKPT_APISR_*。')

    # 相对路径转为绝对路径
    if not os.path.isabs(ckpt_path):
        ckpt_path = os.path.join(PROJECT_DIR, ckpt_path)

    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'权重不存在: {ckpt_path}')

    ckpt = torch.load(ckpt_path, map_location=device)
    if 'params_ema' in ckpt:
        state = ckpt['params_ema']
    elif 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
    else:
        state = ckpt

    model_state = model.state_dict()
    compatible = {k: v for k, v in state.items() if k in model_state and model_state[k].shape == v.shape}
    model.load_state_dict({**model_state, **compatible}, strict=False)

    if is_apisr:
        print(f'  加载 APISR: {ckpt_path}, 兼容 {len(compatible)}/{len(model_state)}')
    else:
        print(f'  加载自训练: {ckpt_path}, Epoch {ckpt.get("epoch", "?")}')

    return model


def load_grl_model(scale: int, ckpt_ours: str | None = None):
    """加载 GRL 2x 模型（与 train_grl_independent 配置一致）。"""
    apisr_tools = os.path.join(PROJECT_DIR, 'APISR_tools')
    if apisr_tools not in sys.path:
        sys.path.insert(0, apisr_tools)
    from architecture.grl import GRL
    model = GRL(
        upscale=scale,
        img_size=112,
        window_size=8,
        depths=[4, 4, 4, 4],
        embed_dim=128,
        num_heads_window=[2, 2, 2, 2],
        num_heads_stripe=[2, 2, 2, 2],
        mlp_ratio=2,
        qkv_proj_type="linear",
        anchor_proj_type="avgpool",
        anchor_window_down_factor=2,
        out_proj_type="linear",
        conv_type="1conv",
        upsampler="pixelshuffle",
    ).to(device)
    model.eval()
    ckpt_path = os.path.join(PROJECT_DIR, ckpt_ours) if ckpt_ours and not os.path.isabs(ckpt_ours) else ckpt_ours
    if not ckpt_path or not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'GRL 权重不存在: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state, strict=True)
    print(f'  加载 GRL: {ckpt_path}, Epoch {ckpt.get("epoch", "?")}')
    return model


In [3]:
# ============================================================
# 2. 推理函数（支持 2x/4x + 大图分块）
# ============================================================


def sr_infer_single(model, lr_img_bgr: np.ndarray, scale: int) -> np.ndarray:
    """对单张 LR 图像做超分，返回 RGB uint8 图像。"""
    lr_img = cv2.cvtColor(lr_img_bgr, cv2.COLOR_BGR2RGB)
    h, w = lr_img.shape[:2]

    # 2x RRDB 有 pixel_unshuffle 要求输入尺寸能被 4 整除
    pad_h = (4 - h % 4) % 4 if scale == 2 else 0
    pad_w = (4 - w % 4) % 4 if scale == 2 else 0
    lr_padded = lr_img
    if pad_h > 0 or pad_w > 0:
        lr_padded = cv2.copyMakeBorder(lr_img, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)

    lr_t = torch.from_numpy(lr_padded.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, _, hh, ww = lr_t.shape
            if hh * ww > 512 * 512:
                h_half, w_half = hh // 2, ww // 2
                sr_t = torch.zeros((1, 3, hh * scale, ww * scale), device=device)
                sr_t[:, :, :h_half*scale, :w_half*scale] = model(lr_t[:, :, :h_half, :w_half])
                sr_t[:, :, :h_half*scale, w_half*scale:] = model(lr_t[:, :, :h_half, w_half:])
                sr_t[:, :, h_half*scale:, :w_half*scale] = model(lr_t[:, :, h_half:, :w_half])
                sr_t[:, :, h_half*scale:, w_half*scale:] = model(lr_t[:, :, h_half:, w_half:])
            else:
                sr_t = model(lr_t)

    sr = sr_t.squeeze(0).float().cpu().clamp(0, 1).numpy().transpose(1, 2, 0)
    sr = sr[:h * scale, :w * scale, :]
    sr_uint8 = (sr * 255.0).round().astype(np.uint8)
    return sr_uint8


def sr_infer_single_grl(model, lr_img_bgr: np.ndarray, scale: int) -> np.ndarray:
    """GRL 单张推理，返回 RGB uint8。"""
    lr_img = cv2.cvtColor(lr_img_bgr, cv2.COLOR_BGR2RGB)
    h, w = lr_img.shape[:2]
    lr_t = torch.from_numpy(lr_img.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, _, hh, ww = lr_t.shape
            if hh * ww > 512 * 512:
                h_half, w_half = hh // 2, ww // 2
                sr_t = torch.zeros((1, 3, hh * scale, ww * scale), device=device)
                sr_t[:, :, :h_half*scale, :w_half*scale] = model(lr_t[:, :, :h_half, :w_half])
                sr_t[:, :, :h_half*scale, w_half*scale:] = model(lr_t[:, :, :h_half, w_half:])
                sr_t[:, :, h_half*scale:, :w_half*scale] = model(lr_t[:, :, h_half:, :w_half])
                sr_t[:, :, h_half*scale:, w_half*scale:] = model(lr_t[:, :, h_half:, w_half:])
            else:
                sr_t = model(lr_t)
    sr = sr_t.squeeze(0).float().cpu().clamp(0, 1).numpy().transpose(1, 2, 0)
    sr = sr[:h * scale, :w * scale, :]
    return (sr * 255.0).round().astype(np.uint8)


def ensure_dir(path: str):
    if path and not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)


In [4]:
# ============================================================
# 3. 全参考指标：PSNR / SSIM / LPIPS / DISTS / 线条 F1（多模型循环）
# ============================================================

lpips_model = lpips.LPIPS(net='vgg').to(device)
lpips_model.eval()

# DISTS 指标（来自 piq）
import piq
try:
    dists_model = piq.DISTS().to(device)
    dists_model.eval()
except Exception as e:
    print('DISTS 初始化失败，请确认已安装 piq:', e)
    dists_model = None


def img_to_tensor_lpips(img_rgb: np.ndarray) -> torch.Tensor:
    """将 [0,255] RGB 转为 LPIPS 所需的 [-1,1] NCHW tensor"""
    x = img_rgb.astype(np.float32) / 255.0
    x = x * 2.0 - 1.0
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    return x.to(device)


def img_to_tensor_dists(img_rgb: np.ndarray) -> torch.Tensor:
    """DISTS 使用 [0,1] 归一化张量。"""
    x = img_rgb.astype(np.float32) / 255.0
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    return x.to(device)


def compute_lpips_patchwise(hr_rgb: np.ndarray, sr_rgb: np.ndarray, patch_size: int = 256, batch_size: int = 4) -> float:
    """将整图切成 patch，分 batch 计算 LPIPS，最后取平均。"""
    h, w = hr_rgb.shape[:2]
    ps = patch_size
    lpips_vals = []

    for y in range(0, h, ps):
        for x in range(0, w, ps):
            hr_patch = hr_rgb[y:y+ps, x:x+ps, :]
            sr_patch = sr_rgb[y:y+ps, x:x+ps, :]
            if hr_patch.size == 0 or sr_patch.size == 0:
                continue
            # 边缘 patch 可能不足 patch_size，这里做反射 padding，保证所有 patch 尺寸一致
            ph, pw = hr_patch.shape[:2]
            pad_h = ps - ph
            pad_w = ps - pw
            if pad_h > 0 or pad_w > 0:
                hr_patch = cv2.copyMakeBorder(hr_patch, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
                sr_patch = cv2.copyMakeBorder(sr_patch, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
            hr_t = img_to_tensor_lpips(hr_patch)
            sr_t = img_to_tensor_lpips(sr_patch)
            lpips_vals.append((hr_t, sr_t))

    scores = []
    with torch.no_grad():
        for i in range(0, len(lpips_vals), batch_size):
            batch = lpips_vals[i:i+batch_size]
            hr_batch = torch.cat([h for h, _ in batch], dim=0)
            sr_batch = torch.cat([s for _, s in batch], dim=0)
            val = lpips_model(hr_batch, sr_batch)
            scores.extend(val.view(-1).cpu().tolist())

    return float(np.mean(scores)) if scores else float('nan')


def compute_dists_patchwise(hr_rgb: np.ndarray, sr_rgb: np.ndarray, patch_size: int = 256, batch_size: int = 4) -> float:
    """将整图切成 patch，分 batch 计算 DISTS，最后取平均。"""
    if dists_model is None:
        return float('nan')

    h, w = hr_rgb.shape[:2]
    ps = patch_size
    pairs = []

    for y in range(0, h, ps):
        for x in range(0, w, ps):
            hr_patch = hr_rgb[y:y+ps, x:x+ps, :]
            sr_patch = sr_rgb[y:y+ps, x:x+ps, :]
            if hr_patch.size == 0 or sr_patch.size == 0:
                continue
            # 与 LPIPS 一样，对边缘不足 patch_size 的 patch 做反射 padding
            ph, pw = hr_patch.shape[:2]
            pad_h = ps - ph
            pad_w = ps - pw
            if pad_h > 0 or pad_w > 0:
                hr_patch = cv2.copyMakeBorder(hr_patch, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
                sr_patch = cv2.copyMakeBorder(sr_patch, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
            hr_t = img_to_tensor_dists(hr_patch)
            sr_t = img_to_tensor_dists(sr_patch)
            pairs.append((hr_t, sr_t))

    scores = []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            hr_batch = torch.cat([h for h, _ in batch], dim=0)
            sr_batch = torch.cat([s for _, s in batch], dim=0)
            val = dists_model(hr_batch, sr_batch)
            scores.extend(val.view(-1).cpu().tolist())

    return float(np.mean(scores)) if scores else float('nan')


def canny_f1(sr_gray: np.ndarray, hr_gray: np.ndarray, t1: int = 100, t2: int = 200) -> float:
    """基于 Canny 边缘的 F1 指标。输入为 uint8 灰度图。"""
    sr_edges = cv2.Canny(sr_gray, t1, t2)
    hr_edges = cv2.Canny(hr_gray, t1, t2)
    sr_bin = sr_edges > 0
    hr_bin = hr_edges > 0
    tp = np.logical_and(sr_bin, hr_bin).sum()
    fp = np.logical_and(sr_bin, ~hr_bin).sum()
    fn = np.logical_and(~sr_bin, hr_bin).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return float(f1)


def _json_default(obj):
    """让 json.dump 能序列化 numpy 标量。"""
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f'Object of type {obj.__class__.__name__} is not JSON serializable')


hr_dir_abs = HR_DIR
all_summaries = []

# 只评估 test_list.txt 中的图像
with open(TEST_LIST_PATH, 'r', encoding='utf-8') as f:
    test_set = set(line.strip() for line in f if line.strip())
print(f'测试集名单共 {len(test_set)} 张图（来自 test_list.txt）')

overall_start = time.time()

# 从第 START_MODEL_INDEX 个模型开始评估
for cfg in configs_to_run:
    model_name = cfg['model_name']
    scale = cfg['scale']
    is_grl = cfg.get('model_type') == 'grl'
    num_block = cfg.get('num_block', 6)
    is_apisr = cfg.get('is_apisr', False)
    lr_dir = os.path.join(PROJECT_DIR, cfg['lr_dir'])
    sr_dir = os.path.join(RESULTS_DIR, f"{model_name}_fullref_sr")
    ensure_dir(sr_dir)
    ensure_dir(RESULTS_DIR)

    print(f"\n>>> 开始评估模型: {model_name} (scale={scale}x, blocks={num_block}, is_apisr={is_apisr}, is_grl={is_grl})")
    model_start = time.time()
    load_start = time.time()
    if is_grl:
        model = load_grl_model(scale, cfg.get('ckpt_ours'))
    else:
        model = load_rrdb_model(scale, num_block, is_apisr, cfg.get('ckpt_ours'))
    load_time = time.time() - load_start

    lr_paths = sorted(glob.glob(os.path.join(lr_dir, '*.[jp][pn]*g')))
    lr_paths = [p for p in lr_paths if os.path.basename(p) in test_set]
    if MAX_IMAGES is not None:
        lr_paths = lr_paths[:MAX_IMAGES]
        print(f'[试运行] 仅评估前 {MAX_IMAGES} 张')
    print(f'[{model_name}] LR 图像数: {len(lr_paths)}（测试集内）')

    psnr_list, ssim_list, lpips_list, f1_list, dists_list = [], [], [], [], []
    per_image_scores = {}

    # 分阶段计时（单位：秒）
    sr_time_total = 0.0
    lpips_time_total = 0.0
    dists_time_total = 0.0
    edge_time_total = 0.0

    for lr_path in tqdm(lr_paths, desc=model_name):
        fname = os.path.basename(lr_path)
        stem, _ext = os.path.splitext(fname)
        hr_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.JPG', '.JPEG']:
            cand = os.path.join(hr_dir_abs, stem + ext)
            if os.path.isfile(cand):
                hr_path = cand
                break
        if hr_path is None:
            continue

        hr_bgr = cv2.imread(hr_path)
        if hr_bgr is None:
            continue
        hr_rgb = cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB)

        lr_bgr = cv2.imread(lr_path)
        if lr_bgr is None:
            continue

        t0 = time.time()
        if is_grl:
            sr_rgb = sr_infer_single_grl(model, lr_bgr, scale)
        else:
            sr_rgb = sr_infer_single(model, lr_bgr, scale)
        sr_time_total += time.time() - t0

        cv2.imwrite(os.path.join(sr_dir, fname), cv2.cvtColor(sr_rgb, cv2.COLOR_RGB2BGR))

        h_hr, w_hr = hr_rgb.shape[:2]
        h_sr, w_sr = sr_rgb.shape[:2]
        if (h_hr != h_sr) or (w_hr != w_sr):
            sr_rgb = cv2.resize(sr_rgb, (w_hr, h_hr), interpolation=cv2.INTER_CUBIC)

        hr_f = hr_rgb.astype(np.float32) / 255.0
        sr_f = sr_rgb.astype(np.float32) / 255.0

        mse = np.mean((hr_f - sr_f) ** 2)
        psnr_val = float('inf') if mse == 0 else (10 * log10(1.0 / mse))
        ssim_val = ssim(hr_f, sr_f, channel_axis=2, data_range=1.0)

        # LPIPS / DISTS 使用 patch + batch 方式计算，避免整图占用显存
        t_lpips = time.time()
        lpips_val = compute_lpips_patchwise(hr_rgb, sr_rgb, patch_size=256, batch_size=4)
        lpips_time_total += time.time() - t_lpips

        t_dists = time.time()
        dists_val = compute_dists_patchwise(hr_rgb, sr_rgb, patch_size=256, batch_size=4)
        dists_time_total += time.time() - t_dists

        t_edge = time.time()
        hr_gray = cv2.cvtColor(hr_rgb, cv2.COLOR_RGB2GRAY)
        sr_gray = cv2.cvtColor(sr_rgb, cv2.COLOR_RGB2GRAY)
        f1_val = canny_f1(sr_gray, hr_gray)
        edge_time_total += time.time() - t_edge

        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        lpips_list.append(lpips_val)
        dists_list.append(dists_val)
        f1_list.append(f1_val)
        per_image_scores[fname] = {
            'PSNR': round(psnr_val, 4),
            'SSIM': round(ssim_val, 4),
            'LPIPS': round(lpips_val, 4),
            'DISTS': round(dists_val, 4) if not np.isnan(dists_val) else None,
            'EdgeF1': round(f1_val, 4),
        }

    n = len(psnr_list)
    model_time = time.time() - model_start
    print(f'[{model_name}] 有效样本: {n}')
    if n == 0:
        del model
        torch.cuda.empty_cache()
        continue

    # 分阶段用时统计
    if n > 0:
        print(f'[{model_name}] 用时统计:')
        print(f'  加载模型: {load_time:.2f} s')
        print(f'  总时间:   {model_time:.2f} s  (~{model_time / n:.3f} s/图)')
        print(f'    ├─ SR 推理:    {sr_time_total:.2f} s  (~{sr_time_total / n:.3f} s/图)')
        print(f'    ├─ LPIPS:      {lpips_time_total:.2f} s  (~{lpips_time_total / n:.3f} s/图)')
        print(f'    ├─ DISTS:      {dists_time_total:.2f} s  (~{dists_time_total / n:.3f} s/图)')
        print(f'    └─ Edge F1:    {edge_time_total:.2f} s  (~{edge_time_total / n:.3f} s/图)')

    avg_psnr = float(np.mean(psnr_list))
    avg_ssim = float(np.mean(ssim_list))
    avg_lpips = float(np.mean(lpips_list))
    avg_dists = float(np.mean(dists_list)) if dists_list else float('nan')
    avg_f1 = float(np.mean(f1_list))
    all_summaries.append({
        'model_name': model_name,
        'PSNR': avg_psnr,
        'SSIM': avg_ssim,
        'LPIPS': avg_lpips,
        'DISTS': avg_dists,
        'EdgeF1': avg_f1,
        'num_images': n,
    })

    summary = {
        'model': model_name,
        'scale': scale,
        'num_block': num_block if not is_grl else None,
        'is_apisr': is_apisr,
        'is_grl': is_grl,
        'num_images': n,
        'metrics': {
            'PSNR': round(avg_psnr, 4),
            'SSIM': round(avg_ssim, 4),
            'LPIPS': round(avg_lpips, 4),
            'DISTS': round(avg_dists, 4) if not np.isnan(avg_dists) else None,
            'EdgeF1': round(avg_f1, 4),
        },
        'per_image': per_image_scores,
    }
    json_path = os.path.join(RESULTS_DIR, f'{model_name}_fullref_eval.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False, default=_json_default)
    txt_path = os.path.join(RESULTS_DIR, f'{model_name}_fullref_eval.txt')
    with open(txt_path, 'w', encoding='utf-8') as f:
        blocks_str = f', blocks={num_block}' if not is_grl else ''
        f.write(f'{model_name} (scale={scale}x{blocks_str}, is_apisr={is_apisr})\n')
        f.write(f'num_images = {n}\n' + '='*40 + '\n')
        f.write(f'PSNR   (↑, dB)     : {avg_psnr:.4f}\n')
        f.write(f'SSIM   (↑)         : {avg_ssim:.4f}\n')
        f.write(f'LPIPS  (↓, VGG)    : {avg_lpips:.4f}\n')
        dists_str = f'{avg_dists:.4f}' if (dists_list and not np.isnan(avg_dists)) else 'nan'
        f.write(f'DISTS  (↓)         : {dists_str}\n')
        f.write(f'Edge F1 (↑, Canny) : {avg_f1:.4f}\n' + '='*40 + '\n')
    print(f'  已保存: {json_path}, {txt_path}')

    del model
    torch.cuda.empty_cache()

print('所有模型评估完成。')


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\admin\.conda\envs\ece284fa25\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\admin\.conda\envs\ece284fa25\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\admin\.conda\envs\ece284fa25\Lib\site-packages\lpips\weights\v0.1\vgg.pth
测试集名单共 434 张图（来自 test_list.txt）

>>> 开始评估模型: APISR_RRDB_2x (scale=2x, blocks=6, is_apisr=True)
  加载 APISR: d:\lixinqi\UCSD\ECE285\AWM\pretrained_models/2x_APISR_RRDB_GAN_generator.pth, 兼容 192/192
[APISR_RRDB_2x] LR 图像数: 434（测试集内）


APISR_RRDB_2x:   0%|          | 2/434 [00:11<41:33,  5.77s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 4. 汇总表：多模型对比（PSNR / SSIM / LPIPS / DISTS / Edge F1）
# ============================================================

if not all_summaries:
    print('⚠ 无有效评估结果，请先运行上一 cell 并检查 LR/HR 路径与 MODEL_CONFIGS。')
else:
    import numpy as np

    print('\n' + '='*80)
    print('  全参考评估汇总（↑ 越高越好：PSNR / SSIM / Edge F1；↓ 越低越好：LPIPS / DISTS）')
    print('='*80)
    fmt = '{:<30} {:>9} {:>9} {:>9} {:>9} {:>9}'
    print(fmt.format('Model', 'PSNR↑', 'SSIM↑', 'LPIPS↓', 'DISTS↓', 'EdgeF1↑'))
    print('-'*80)
    for s in all_summaries:
        dists_str = f"{s['DISTS']:.4f}" if not np.isnan(s['DISTS']) else '  nan'
        print(fmt.format(
            s['model_name'][:29],
            f"{s['PSNR']:.4f}",
            f"{s['SSIM']:.4f}",
            f"{s['LPIPS']:.4f}",
            dists_str,
            f"{s['EdgeF1']:.4f}",
        ))
    print('='*80)
    print('✅ 各模型 JSON/TXT 已保存至 results 目录，汇总表如上。')



  全参考评估汇总（↑ 越高越好：PSNR / SSIM / Edge F1；↓ 越低越好：LPIPS / DISTS）
Model                              PSNR↑     SSIM↑    LPIPS↓    DISTS↓   EdgeF1↑
--------------------------------------------------------------------------------
APISR_RRDB_2x                    23.7874    0.8179    0.2552    0.1948    0.5268
✅ 各模型 JSON/TXT 已保存至 results 目录，汇总表如上。
